# Chunking Benchmark trên Kaggle

Notebook này là lớp điều phối mỏng cho framework `chunkbench`: lấy code, chọn dữ liệu, kiểm tra adapter, chạy benchmark và xuất artifact. Không đặt implementation của chunker, metric hay adapter trong notebook.

Trước khi chạy: bật **Internet** trong Kaggle Settings. Với benchmark đầy đủ, hãy thêm dữ liệu theo đúng raw schema của từng adapter qua **Add Data** hoặc cung cấp một Hugging Face repository đã có layout adapter-ready. HotpotQA FullWiki cần corpus Wikipedia đã xử lý; ViMQA cần dữ liệu được cấp quyền.

## 0. Cấu hình chung

Chỉ sửa cell này khi đổi tập dữ liệu, chế độ chạy hoặc commit cần tái lập. `smoke` dùng fixture có sẵn; `full` dùng dữ liệu thật đã được chuẩn bị ở bước 2. Mặc định chỉ chạy bốn baseline publishable; không trộn kết quả mock vào bảng chính.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ManhTanTran/kaggle-notebook.git"
REPO_REVISION = "main"  # Nên thay bằng commit SHA khi chốt kết quả nghiên cứu.
PROJECT_DIR = Path("/kaggle/working/chunking-benchmark")

RUN_MODE = "smoke"  # smoke | full
RESUME = True
FAIL_FAST = False
SEED = 42
MIN_MAPPING_RATE = 0.95

SELECTED_DATASETS = [
    "qasper",
    "hotpotqa_fullwiki",
    "uit_viquad",
    "vimqa",
]

# Bốn baseline này có thể báo cáo. Advanced methods chỉ thêm vào sau
# khi đã cấu hình real backend/model tương ứng, không dùng backend=mock.
SELECTED_METHODS = [
    "fixed_256",
    "fixed_512",
    "sentence_fixed_256",
    "sentence_fixed_512",
]

OUTPUT_ROOT = Path("/kaggle/working/benchmark_outputs")

# kaggle_input: khuyến nghị cho benchmark đầy đủ/lặp lại.
# huggingface_snapshot: chỉ dùng khi mỗi repo HF đã chứa đúng raw schema
# mà DatasetAdapter của framework yêu cầu; notebook không tự suy diễn evidence.
DATA_MODE = "kaggle_input"  # kaggle_input | huggingface_snapshot

DATA_ROOTS = {
    "qasper": Path("/kaggle/input/qasper"),
    "hotpotqa_fullwiki": Path("/kaggle/input/hotpotqa-fullwiki"),
    "uit_viquad": Path("/kaggle/input/uit-viquad"),
    "vimqa": Path("/kaggle/input/vimqa"),
}

# Điền repo HF adapter-ready nếu dùng DATA_MODE = huggingface_snapshot.
# Để None nếu chưa có; các repo công khai đã chuyển đổi schema cần được
# kiểm tra bằng adapter validation ở bước 2.
HF_REPOSITORIES = {
    "qasper": None,
    "hotpotqa_fullwiki": None,
    "uit_viquad": None,
    "vimqa": None,
}


## 1. Clone, cài đặt và kiểm tra môi trường

In [ ]:
import importlib
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "fetch", "--tags", "origin"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", REPO_REVISION],
    check=True,
)
branch = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "symbolic-ref", "--quiet", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
if branch:
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", branch],
        check=True,
    )
os.chdir(PROJECT_DIR)

# Baseline + notebook tooling. Đổi thành .[dev,advanced] nếu chạy advanced real methods.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    check=True,
)

git_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()

chunkbench = importlib.import_module("chunkbench")

print("Project :", PROJECT_DIR)
print("Commit  :", git_commit)
print("Python  :", sys.version.split()[0])
print("Package :", chunkbench.__file__)

try:
    import torch
    print("PyTorch :", torch.__version__)
    print("CUDA    :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch chưa được cài; không cần cho baseline.")


## 2. Xác định/tải và xác thực dữ liệu

Ở chế độ `kaggle_input`, thêm từng dataset tại **Add Data** rồi chỉnh `DATA_ROOTS`. Mỗi thư mục phải theo raw schema đã mô tả ở `docs/datasets/`.

- QASPER: thư mục chứa `validation.json` dạng paper-id keyed JSON.
- HotpotQA FullWiki: `validation.json` và `corpus/` chứa Wikipedia đã sentence-segmented.
- UIT-ViQuAD: thư mục chứa `validation.json` kiểu SQuAD.
- ViMQA: thư mục chứa `dev.json` hoặc `validation.json` đã được cấp quyền.

Hugging Face snapshot chỉ là cơ chế tải file. Nó không bảo đảm schema/evidence tương thích; validation dưới đây là cổng bắt buộc trước benchmark.

In [ ]:
import shutil

if RUN_MODE == "full" and DATA_MODE == "huggingface_snapshot":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
        check=True,
    )
    from huggingface_hub import snapshot_download

    for dataset_name in SELECTED_DATASETS:
        repository_id = HF_REPOSITORIES[dataset_name]
        if not repository_id:
            raise ValueError(
                f"Thiếu HF_REPOSITORIES[{dataset_name!r}]. Dùng Kaggle Input "
                "hoặc điền repo adapter-ready."
            )
        target = PROJECT_DIR / "data" / "raw" / dataset_name
        snapshot_download(repo_id=repository_id, repo_type="dataset", local_dir=target)
        DATA_ROOTS[dataset_name] = target

if RUN_MODE == "full":
    missing = [name for name in SELECTED_DATASETS if not DATA_ROOTS[name].exists()]
    if missing:
        raise FileNotFoundError(
            "Chưa tìm thấy Kaggle Input/dataset snapshot: " + ", ".join(missing)
        )

for name in SELECTED_DATASETS:
    print(f"{name}: {DATA_ROOTS[name]}")


In [ ]:
import yaml

from chunkbench.data.validation import validate_dataset
from chunkbench.registry.datasets import build_dataset_adapter

BASE_CONFIG = (
    Path("configs/experiments/all_qa_datasets_smoke.yaml")
    if RUN_MODE == "smoke"
    else Path("configs/experiments/all_qa_datasets_core_methods.yaml")
)
runtime_dir = PROJECT_DIR / "runtime_configs"
runtime_dir.mkdir(exist_ok=True)
experiment = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
dataset_items = experiment["datasets"]
runtime_dataset_configs = []
validation_reports = {}

for item in dataset_items:
    source_path = PROJECT_DIR / item
    dataset_config = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    dataset_name = dataset_config["name"]
    if dataset_name not in SELECTED_DATASETS:
        continue

    if RUN_MODE == "full":
        root = DATA_ROOTS[dataset_name]
        if dataset_name == "hotpotqa_fullwiki":
            dataset_config["questions_path"] = str(root / "validation.json")
            dataset_config["corpus_path"] = str(root / "corpus")
        else:
            dataset_config["data_path"] = str(root)

    adapter_name = dataset_config.get("adapter", dataset_name)
    bundle = build_dataset_adapter(adapter_name, dataset_config).load()
    report = validate_dataset(bundle, dataset_config.get("validation"))
    mapping_rate = report["evidence_mapping_rate"]
    if mapping_rate < MIN_MAPPING_RATE:
        raise RuntimeError(
            f"{dataset_name}: mapping rate {mapping_rate:.2%} < {MIN_MAPPING_RATE:.2%}"
        )

    target_path = runtime_dir / f"{dataset_name}.yaml"
    target_path.write_text(
        yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    runtime_dataset_configs.append(str(target_path))
    validation_reports[dataset_name] = report
    print(
        f"{dataset_name}: documents={report['document_count']}, "
        f"queries={report['query_count']}, evidence={report['evidence_count']}, "
        f"mapping={mapping_rate:.2%}"
    )

if not runtime_dataset_configs:
    raise RuntimeError("Không có dataset nào được chọn.")


## 3. Chạy smoke hoặc full benchmark

Cell này tạo một YAML runtime ở `/kaggle/working/chunking-benchmark/runtime_configs/`, không sửa config gốc trong repository. Nếu `RUN_MODE = full`, dữ liệu thật đã qua validation ở bước trước mới được chạy.

In [ ]:
methods = [
    spec for spec in experiment["methods"]
    if spec["name"] in SELECTED_METHODS
]
if len(methods) != len(SELECTED_METHODS):
    available = [spec["name"] for spec in experiment["methods"]]
    raise ValueError(f"Method không có trong profile hiện tại. Có: {available}")

# Không cho mock backend lọt vào một full benchmark có thể báo cáo.
if RUN_MODE == "full":
    mocked = [
        spec["name"] for spec in methods
        if spec.get("backend_type") == "mock"
        or spec.get("representation", {}).get("backend_type") == "mock"
    ]
    if mocked:
        raise ValueError(
            "Full benchmark không nhận mock methods: " + ", ".join(mocked)
        )

run_name = f"kaggle_{RUN_MODE}_{'_'.join(SELECTED_DATASETS)}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
experiment["run_name"] = run_name
experiment["datasets"] = runtime_dataset_configs
experiment["methods"] = methods
experiment["output_dir"] = str(OUTPUT_ROOT)
experiment["seed"] = SEED
experiment.setdefault("execution", {})
experiment["execution"].update({
    "resume": RESUME,
    "skip_completed": RESUME,
    "fail_fast": FAIL_FAST,
})

runtime_experiment = runtime_dir / "experiment.yaml"
runtime_experiment.write_text(
    yaml.safe_dump(experiment, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

command = [sys.executable, "-m", "chunkbench.cli", "--config", str(runtime_experiment)]
log_path = OUTPUT_ROOT / f"{run_name}.log"
print("Lệnh chạy:", " ".join(command))

process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
with log_path.open("w", encoding="utf-8") as log_file:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

if process.wait() != 0:
    raise RuntimeError(f"Benchmark thất bại; xem log: {log_path}")

RUN_DIR = OUTPUT_ROOT / run_name
required_artifacts = [
    RUN_DIR / "benchmark_metrics.csv",
    RUN_DIR / "chunk_statistics.csv",
    RUN_DIR / "experiment_manifest.json",
]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError("Thiếu artifact: " + ", ".join(missing_artifacts))
print(f"Hoàn tất: {RUN_DIR}")


## 4. Đọc, hiển thị và xuất kết quả

In [ ]:
import json

import pandas as pd

metrics_df = pd.read_csv(RUN_DIR / "benchmark_metrics.csv")
chunk_stats_df = pd.read_csv(RUN_DIR / "chunk_statistics.csv")
failed_path = RUN_DIR / "failed_runs.json"
failed_runs = (
    json.loads(failed_path.read_text(encoding="utf-8"))
    if failed_path.exists()
    else []
)

publishable_df = metrics_df.copy()
if "IsMockBackend" in publishable_df:
    publishable_df = publishable_df[
        ~publishable_df["IsMockBackend"].fillna(False)
    ]
if "IsPublishableBenchmark" in publishable_df:
    publishable_df = publishable_df[
        publishable_df["IsPublishableBenchmark"].fillna(False)
    ]

columns = [
    "Dataset", "Method", "Hit@10", "MRR@10",
    "EvidenceRecallMacro@10", "EvidenceCoverage@10",
    "EvidenceRecallMacro@2048Tokens", "EvidenceCoverage@2048Tokens",
    "Redundancy@10", "Tokens per Chunk Mean", "RuntimeSeconds",
]
columns = [column for column in columns if column in publishable_df.columns]

print(
    f"Metric rows: {len(metrics_df)} | "
    f"Chunk-stat rows: {len(chunk_stats_df)} | Failed runs: {len(failed_runs)}"
)
display(
    publishable_df[columns].sort_values(
        ["Dataset", "EvidenceRecallMacro@10"], ascending=[True, False]
    )
)

if failed_runs:
    display(pd.DataFrame(failed_runs))


In [ ]:
required_dataset_count = len(SELECTED_DATASETS)
coverage = publishable_df.groupby("Method")["Dataset"].nunique()
complete_methods = coverage[coverage == required_dataset_count].index
macro_df = publishable_df[publishable_df["Method"].isin(complete_methods)]
macro_columns = [
    "Hit@10", "MRR@10", "EvidenceRecallMacro@10",
    "EvidenceCoverage@10", "Redundancy@10",
]
macro_columns = [column for column in macro_columns if column in macro_df.columns]

if macro_columns and not macro_df.empty:
    display(
        macro_df.groupby("Method")[macro_columns].mean().sort_values(
            "EvidenceRecallMacro@10", ascending=False
        )
    )
else:
    print(
        "Chưa có method hoàn tất trên tất cả dataset được chọn; "
        "không tính macro average."
    )

archive_path = shutil.make_archive(
    base_name=str(Path("/kaggle/working") / f"{run_name}_results"),
    format="zip",
    root_dir=RUN_DIR,
)
print("Đã tạo file tải về từ Kaggle Output:", archive_path)
